# FedAvg From Scratch — IID MNIST Baseline

This notebook is the **IID baseline experiment** for the project. The FedAvg implementation
was developed from scratch in PyTorch and now lives in reusable modules under `src/`.

**Research question.** With communication rounds held fixed, how does increasing the number
of local epochs \(E\) affect:

- global test accuracy and loss,
- communication rounds needed to reach target accuracy,
- cumulative local computation?

The checked-in baseline uses 5 clients, full participation, MNIST, SGD, batch size 64,
learning rate 0.01, and \(E\in\{1,5,10\}\).

> The expensive full sweep is disabled by default. The saved baseline results under
> `results/iid_baseline/` are loaded for analysis.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

def find_repo_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    colab_candidate = Path("/content/federated-learning-under-heterogeneity")
    if colab_candidate.exists():
        candidates.append(colab_candidate)

    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate

    raise RuntimeError(
        "Repository root not found. Open this notebook from the repository, "
        "or cd into /content/federated-learning-under-heterogeneity in Colab."
    )

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
import copy
import random
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.models import SimpleMLP
from src.data import (
    partition_iid,
    partition_shard,
    create_client_loaders,
    client_label_counts,
)
from src.training import (
    evaluate,
    client_update,
    federated_train,
)
from src.metrics import model_distance

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_best_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_mnist(root):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    train_ds = datasets.MNIST(root=root, train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(root=root, train=False, download=True, transform=transform)
    return train_ds, test_ds


def verify_partition(client_indices, dataset_size: int):
    flat = [
        int(idx)
        for indices in client_indices.values()
        for idx in indices
    ]
    assert len(flat) == dataset_size, "Some examples were dropped or duplicated."
    assert len(set(flat)) == dataset_size, "Client partitions overlap."
    assert min(flat) >= 0 and max(flat) < dataset_size
    return {
        "total_assigned": len(flat),
        "unique_assigned": len(set(flat)),
    }

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    num_clients: int = 5
    num_rounds: int = 20
    batch_size: int = 64
    learning_rate: float = 0.01
    local_epoch_values: tuple = (1, 5, 10)
    datasets_dir: str = "datasets"
    results_dir: str = "results/iid_baseline"


CFG = ExperimentConfig()
device = get_best_device() if "get_best_device" in globals() else torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

### Implementation map

The reusable implementation is intentionally outside the notebook:

- `src/models.py` — `SimpleMLP`
- `src/data.py` — client partitioning and data loaders
- `src/aggregate.py` — weighted FedAvg aggregation
- `src/training.py` — local training, evaluation, and federated loop
- `src/metrics.py` — model-distance diagnostics

The notebook now focuses on **experimental design and analysis**, rather than duplicating
the implementation.

## 2. Load MNIST and construct IID clients

In [ ]:
set_seed(CFG.seed)

train_ds, test_ds = load_mnist(REPO_ROOT / CFG.datasets_dir)

client_indices = partition_iid(
    train_ds,
    num_clients=CFG.num_clients,
    seed=CFG.seed,
)
partition_check = verify_partition(client_indices, len(train_ds))

client_loaders = create_client_loaders(
    train_ds,
    client_indices,
    batch_size=CFG.batch_size,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
)

print(partition_check)
client_label_counts(train_ds, client_indices)

With 60,000 MNIST training examples and 5 clients, each client receives 12,000 examples.
Because the assignment is IID, the digit distribution should be approximately similar
across clients.

## 3. Controlled initialization

In [ ]:
INITIAL_MODEL_PATH = REPO_ROOT / "results/iid_baseline/initial_model.pt"

initial_state = torch.load(INITIAL_MODEL_PATH, map_location="cpu")

global_model = SimpleMLP(784, 10)
global_model.load_state_dict(initial_state)
global_model.to(device)

loss_fn = nn.CrossEntropyLoss()
initial_loss, initial_acc = evaluate(global_model, test_loader, loss_fn, device)

print(f"Initial loss: {initial_loss:.6f}")
print(f"Initial accuracy: {initial_acc:.4f}")

The saved checkpoint is the common \(w_0\) used for controlled comparisons. The expected
baseline evaluation is approximately **loss 2.313339, accuracy 12.67%**.

## 4. FedAvg reminder

In every communication round:

1. each participating client receives the same current global model \(w_t\);
2. each client trains locally for \(E\) epochs;
3. the server aggregates the resulting models.

For client \(k\) with \(n_k\) examples,

\[
w_{t+1}
=
\sum_k
\frac{n_k}{\sum_j n_j}
w_{t+1}^{(k)}.
\]

Here every IID client has 12,000 examples, so the weights are all \(1/5\). The implementation
remains sample-weighted so it also supports later quantity-skew experiments.

## 5. Optional reproduction of the \(E\)-sweep

In [ ]:
def run_e_sweep(
    initial_state,
    client_loaders,
    test_loader,
    e_values,
    num_rounds,
    learning_rate,
    device,
):
    histories = {}
    trained_models = {}
    runtimes = {}

    for E in e_values:
        print(f"Running E={E}...")
        set_seed(CFG.seed)

        model = SimpleMLP(784, 10)
        model.load_state_dict(copy.deepcopy(initial_state))
        model.to(device)

        start = time.perf_counter()
        trained_model, history_df = federated_train(
            global_model=model,
            client_loaders=client_loaders,
            num_rounds=num_rounds,
            local_epochs=E,
            learning_rate=learning_rate,
            loss_fn=nn.CrossEntropyLoss(),
            test_loader=test_loader,
            device=device,
            verbose=True,
        )
        runtimes[E] = time.perf_counter() - start
        histories[E] = history_df
        trained_models[E] = trained_model

    return histories, trained_models, runtimes


RUN_FULL_SWEEP = False

if RUN_FULL_SWEEP:
    histories, trained_models, runtimes = run_e_sweep(
        initial_state=initial_state,
        client_loaders=client_loaders,
        test_loader=test_loader,
        e_values=CFG.local_epoch_values,
        num_rounds=CFG.num_rounds,
        learning_rate=CFG.learning_rate,
        device=device,
    )
else:
    print("Full sweep skipped; loading saved IID results below.")

## 6. Saved IID baseline results

The checked-in baseline currently uses one seed (`42`), so these results are a controlled
baseline rather than multi-seed research estimates.

| E | Final test accuracy | Rounds to 90% | Local epochs to 90% | Rounds to 95% | Local epochs to 95% |
|---:|---:|---:|---:|---:|---:|
| 1 | 92.33% | 7 | 7 | — | — |
| 5 | 96.29% | 2 | 10 | 12 | 60 |
| 10 | 97.25% | 1 | 10 | 6 | 60 |

**Interpretation.** Larger \(E\) improves convergence per communication round under IID data,
but it also increases local computation. In particular, \(E=5\) and \(E=10\) both require
about 60 cumulative local epochs per client to reach 95%, while \(E=10\) uses fewer
communication rounds.

In [ ]:
RESULT_DIR = REPO_ROOT / CFG.results_dir

saved_histories = {
    E: pd.read_csv(RESULT_DIR / f"fedavg_iid_E{E}_history.csv")
    for E in CFG.local_epoch_values
}
summary_df = pd.read_csv(RESULT_DIR / "summary.csv")

summary_df

In [ ]:
plt.figure(figsize=(8, 5))
for E, history_df in saved_histories.items():
    plt.plot(
        history_df["round"],
        history_df["test_accuracy"],
        marker="o",
        label=f"E={E}",
    )

plt.xlabel("Communication Round")
plt.ylabel("Test Accuracy")
plt.title("FedAvg under IID data: accuracy vs communication")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for E, history_df in saved_histories.items():
    plt.plot(
        history_df["round"] * E,
        history_df["test_accuracy"],
        marker="o",
        label=f"E={E}",
    )

plt.xlabel("Cumulative Local Epochs per Client")
plt.ylabel("Test Accuracy")
plt.title("FedAvg under IID data: accuracy vs local computation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Limitations and next experiment

**Current limitations**

- one random seed,
- IID clients only,
- full client participation (\(C=1\)),
- cumulative local epochs are only a proxy for computation,
- the current parameter-wise aggregator is appropriate for this MLP but is not yet a
  general solution for models with persistent buffers such as BatchNorm statistics.

**Next:** hold the model, optimizer, test set, initialization, and training loop fixed while
replacing only the IID client partition with controlled heterogeneous partitions.